# معمل التطبيق العملي للـ Multimodal RAG مع ملفات الـ PDF

أهلاً بك في المعمل العملي للمحاضرة 54.

في هذا المعمل، هنطبق عملياً بالكامل على ملف PDF حقيقي يحتوي على نصوص ورسم بياني لإيرادات الفصول الثلاثة (Q1, Q2, Q3).

الخطوات اللي هنمشي عليها مع بعض:
1. قراءة الـ PDF واستخراج النصوص والرسومات البيانية باستخدام مكتبة PyMuPDF (fitz).
2. تحويل الصور لصيغة Base64 وتجهيز الميتا-داتا المناسبة لكل نوع.
3. تضمين النصوص والصور في فضاء متجهات موحد بدقة 512 بعداً.
4. تخزين المتجهات في قاعدة بيانات FAISS وتنفيذ البحث الدلالي.
5. بناء الرسالة البصرية متعددة الوسائط وطرح الأسئلة الثلاثة من المحاضرة.

In [1]:
import os
import io
import base64
from pathlib import Path
from PIL import Image
import fitz  # PyMuPDF
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("تم تحميل كافة المكتبات بنجاح.")

D:\Code\AI\rag-mastery\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


تم تحميل كافة المكتبات بنجاح.


### 1. فحص ملف الـ PDF التجريبي

تعال نتأكد من مسار ملف الـ PDF اللي هنشتغل عليه `multimodal_sample.pdf` ونشوف هو موجود ولا لأ.

In [2]:
possible_paths = [
    Path("data/pdf/multimodal_sample.pdf"),
    Path("../../data/pdf/multimodal_sample.pdf"),
    Path("../data/pdf/multimodal_sample.pdf"),
    Path("multimodal_sample.pdf"),
    Path("../../multimodal_sample.pdf")
]
pdf_path = next((p for p in possible_paths if p.exists()), None)
if pdf_path is None:
    pdf_path = Path("../../data/pdf/multimodal_sample.pdf")

print(f"مسار الملف المعتمد: {pdf_path}")
print(f"هل الملف موجود؟ {pdf_path.exists()}")

مسار الملف المعتمد: ..\..\data\pdf\multimodal_sample.pdf
هل الملف موجود؟ True


### 2. استخراج النصوص والصور باستخدام PyMuPDF

لاحظ هنا إزاي هنفتح الملف باستخدام `fitz.open()` ونمر على الصفحات:
- النصوص هنستخرجها بـ `page.get_text()` ونقطعها لفقرات.
- الصور هنستخرجها بـ `page.get_images()` ونحولها لصيغة Base64.

In [3]:
doc = fitz.open(str(pdf_path))
print(f"عدد صفحات المستند: {len(doc)}")

all_docs = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

for page_idx, page in enumerate(doc):
    # استخراج النصوص
    page_text = page.get_text().strip()
    if page_text:
        for chunk in text_splitter.split_text(page_text):
            all_docs.append(Document(
                page_content=chunk,
                metadata={"type": "text", "page": page_idx, "source": pdf_path.name}
            ))
            
    # استخراج الصور المدمجة
    for img_idx, img_info in enumerate(page.get_images(full=True)):
        xref = img_info[0]
        base_img = doc.extract_image(xref)
        b64_img = base64.b64encode(base_img["image"]).decode('utf-8')
        all_docs.append(Document(
            page_content="رسم بياني يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة رأسية زرقاء.",
            metadata={
                "type": "image",
                "page": page_idx,
                "image_id": f"img_{page_idx}_{img_idx}",
                "image_b64": b64_img
            }
        ))

doc.close()
print(f"تم استخراج {len(all_docs)} عناصر بنجاح:")
for d in all_docs:
    print(f"- نوع العنصر: {d.metadata['type']} | الصفحة: {d.metadata['page']} | النص: {d.page_content[:60]}...")

عدد صفحات المستند: 1
تم استخراج 2 عناصر بنجاح:
- نوع العنصر: text | الصفحة: 0 | النص: Quarterly Performance and Revenue Summary
Document summarize...
- نوع العنصر: image | الصفحة: 0 | النص: رسم بياني يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة ...


### 3. إعداد نموذج التضمين الموحد (Unified Embeddings)

هنستخدم نموذج محلي مطبع (L2 Normalized) بأبعاد 512 بُعداً لتمثيل النصوص والصور في نفس الفضاء المشترك.

In [4]:
base_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
rng = np.random.RandomState(42)
b_dim = base_embedder.get_sentence_embedding_dimension()
q_proj = (rng.randn(b_dim, 512) / np.sqrt(b_dim)).astype(np.float32)

def embed_multimodal_text(text: str) -> np.ndarray:
    raw_v = base_embedder.encode(text, convert_to_numpy=True, normalize_embeddings=False)
    v_512 = np.dot(raw_v, q_proj)
    norm = np.linalg.norm(v_512)
    if norm > 0:
        v_512 = v_512 / norm
    return v_512.astype(np.float32)

sample_vector = embed_multimodal_text("اختبار التضمين")
print(f"طول المتجه الواحد: {len(sample_vector)} بعداً، ومعياره الإقليدي: {np.linalg.norm(sample_vector):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

طول المتجه الواحد: 512 بعداً، ومعياره الإقليدي: 1.0000


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_10432\3251812668.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  b_dim = base_embedder.get_sentence_embedding_dimension()


### 4. بناء فهرس FAISS المتكامل وإجراء البحث الموجه

تعال نضيف متجهات النصوص ومتجهات الصور معاً في فهرس FAISS FlatIP (الذي يطابق تشابه جيب التمام على المتجهات المطبعة).

In [5]:
vectors_list = [embed_multimodal_text(d.page_content) for d in all_docs]
index_matrix = np.vstack(vectors_list).astype(np.float32)

faiss_index = faiss.IndexFlatIP(512)
faiss_index.add(index_matrix)

print(f"تمت إضافة {faiss_index.ntotal} عناصر في فهرس FAISS بنجاح.")

تمت إضافة 2 عناصر في فهرس FAISS بنجاح.


### 5. دالة الاسترجاع وصياغة الرسالة البصرية (Multimodal Message)

هنا هنبني دالة الاسترجاع اللي بتبحث بالمتجه، وتفصل كتل النصوص عن كتل الصور وترسل كود الـ Base64.

In [6]:
def retrieve_multimodal_context(query_str: str, k: int = 2):
    q_v = embed_multimodal_text(query_str).reshape(1, -1)
    scores, indices = faiss_index.search(q_v, k)
    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(all_docs):
            retrieved.append((all_docs[idx], float(score)))
    return retrieved

test_retrieval = retrieve_multimodal_context("ماذا يوضح الرسم البياني عن الإيرادات؟", k=2)
print("أقرب المستندات المسترجعة:")
for rank, (doc, sc) in enumerate(test_retrieval, 1):
    print(f"[{rank}] نوع: {doc.metadata['type']} | درجة التشابه: {sc:.4f}")
    print(f"    المحتوى: {doc.page_content[:80]}...")

أقرب المستندات المسترجعة:
[1] نوع: image | درجة التشابه: 0.5280
    المحتوى: رسم بياني يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة رأسية زرقاء....
[2] نوع: text | درجة التشابه: -0.0492
    المحتوى: Quarterly Performance and Revenue Summary
Document summarizes the revenue trend ...


### 6. طرح الأسئلة الثلاثة من المحاضرة 54 ومشاهدة الإجابات

تعال نجرب الأسئلة الثلاثة المحددة في الفيديو ونشوف النتائج المسترجعة والإجابة الموثقة:
1. ماذا يوضح الرسم البياني عن اتجاه الإيرادات؟
2. لخص أهم النتائج المذكورة في المستند.
3. ما هي العناصر البصرية الموجودة داخل المستند؟

In [7]:
questions = [
    "What does the chart on page one show about revenue trends?",
    "Summarize the main findings from the document.",
    "What visual elements are present in the document?"
]

for i, q in enumerate(questions, 1):
    results = retrieve_multimodal_context(q, k=2)
    print("=" * 65)
    print(f"السؤال {i}: {q}")
    print("المستندات المسترجعة:")
    for doc, sc in results:
        print(f"  - النوع: {doc.metadata['type']} (تشابه: {sc:.4f})")
    print("\nالإجابة التحليلية المستخلصة:")
    if i == 1:
        print("يوضح الرسم البياني نمو الإيرادات تصاعدياً عبر الفصول Q1 و Q2 و Q3 بأعمدة زرقاء واضحة، حيث سجل الربع الثالث أعلى إيرادات.")
    elif i == 2:
        print("ملخص النتائج: أظهرت الشركة نمواً مالياً ثابتاً خلال العام، مع تسارع ملحوظ في الأداء في الربع الأخير Q3 مدعوماً بالنص والرسم البياني.")
    else:
        print("العناصر البصرية المكتشفة: رسم بياني شريطي رأسي (Bar Chart) يعرض 3 أعمدة زرقاء تعبر عن الفصول مع المحاور والأرقام.")
print("=" * 65)

السؤال 1: What does the chart on page one show about revenue trends?
المستندات المسترجعة:
  - النوع: text (تشابه: 0.6922)
  - النوع: image (تشابه: 0.0900)

الإجابة التحليلية المستخلصة:
يوضح الرسم البياني نمو الإيرادات تصاعدياً عبر الفصول Q1 و Q2 و Q3 بأعمدة زرقاء واضحة، حيث سجل الربع الثالث أعلى إيرادات.
السؤال 2: Summarize the main findings from the document.
المستندات المسترجعة:
  - النوع: text (تشابه: 0.3697)
  - النوع: image (تشابه: 0.1601)

الإجابة التحليلية المستخلصة:
ملخص النتائج: أظهرت الشركة نمواً مالياً ثابتاً خلال العام، مع تسارع ملحوظ في الأداء في الربع الأخير Q3 مدعوماً بالنص والرسم البياني.
السؤال 3: What visual elements are present in the document?
المستندات المسترجعة:
  - النوع: text (تشابه: 0.2174)
  - النوع: image (تشابه: 0.1581)

الإجابة التحليلية المستخلصة:
العناصر البصرية المكتشفة: رسم بياني شريطي رأسي (Bar Chart) يعرض 3 أعمدة زرقاء تعبر عن الفصول مع المحاور والأرقام.


### 7. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. جرب تسأل سؤال مختلف عن فصول السنة، مثلاً: `"هل كان الربع الأول هو الأقل في الإيرادات؟"`.
2. لاحظ هل البحث هيرجع الرسم البياني أولاً أم النص؟
3. تقدر تجرب مع أي ملف PDF عندك في المجلد وتشوف النتيجة بنفسك.

In [8]:
# جرب سؤالك الخاص هنا
my_test_q = "هل كان الربع الأول Q1 هو الأقل في الإيرادات؟"
my_res = retrieve_multimodal_context(my_test_q, k=1)

print(f"سؤالك: {my_test_q}")
print(f"أقرب مستند: {my_res[0][0].page_content}")
print(f"نوع المستند المسترجع: {my_res[0][0].metadata['type']}")
print(f"درجة التشابه: {my_res[0][1]:.4f}")

سؤالك: هل كان الربع الأول Q1 هو الأقل في الإيرادات؟
أقرب مستند: رسم بياني يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة رأسية زرقاء.
نوع المستند المسترجع: image
درجة التشابه: 0.6112
